<a href="https://colab.research.google.com/github/ble27/CSCE-491/blob/main/FP32_INT8_Quantization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exploring FP32 to INT8 Quantization


In [ ]:
!nvidia-smi
!pip install transformers optimum[exporters] torch

Tue Sep 29 22:32:27 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   48C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Shared packages for TensorRT and non-TensorRT pipeline


In [ ]:
# Colab GPU runtime: shared packages for BOTH pipelines
# Do not install/upgrade pandas, numpy, protobuf, torch, or torchvision.

%pip install -q -U "setuptools<82" wheel jedi

%pip install -q -U \
  "onnx>=1.18.0" \
  onnxscript \
  onnxsim \
  onnxruntime \
  "optimum-onnx[onnxruntime]" \
  timm \
  transformers


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 23.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 32.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 754.2/754.2 kB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 41.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 32.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 61.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 30.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.8/185.8 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.2/161.2 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 343.2/343.

### Non-TensorRT download


In [ ]:
# FP32 ONNX export + ONNX Runtime static INT8 PTQ + ORT inference
!pip install -q -U "optimum-onnx[onnxruntime]" onnxruntime

### TensortRT download

In [ ]:
!pip install tensorrt
import tensorrt
print(tensorrt.__version__)

11.3.0.99


In [ ]:
# Non-TensorRT pipeline:
#  - FP32 HF model
#  - Export to ONNX
#  - ONNX runtime PTQ INT8 quantization
# - Benchmark / results

'\nNon-TensorRT pipeline:\n  - FP32 HF model\n  - Export to ONNX\n  - ONNX runtime PTQ INT8 quantization\n  - Benchmark / results\n'

In [ ]:
import sys
import torch
import onnx
import onnxruntime as ort
import timm
import transformers
import onnxscript

print("Python:", sys.version.split()[0])
print("Torch:", torch.__version__)
print("ONNX:", onnx.__version__)
print("ORT:", ort.__version__)
print("timm:", timm.__version__)
print("Transformers:", transformers.__version__)
print("ONNX Script:", onnxscript.__version__)
print("CUDA:", torch.cuda.is_available())

import onnxruntime as ort
print(ort.get_available_providers())

from pathlib import Path

# Create folders to store models
for folder in [
    "models",
    "models/fp32_onnx",
    "models/int8_onnx",
    "data/calibration",
    "data/validation",
    "results",
]:
    Path(folder).mkdir(parents=True, exist_ok=True)

# Save to Google Drive
from google.colab import drive
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/PTQ_MobileNet")

for folder in [
    PROJECT / "models" / "fp32_onnx",
    PROJECT / "models" / "int8_onnx",
    PROJECT / "data" / "calibration",
    PROJECT / "data" / "evaluation",
    PROJECT / "results",
]:
    folder.mkdir(parents=True, exist_ok=True)

FP32_ONNX = PROJECT / "models" / "fp32_onnx" / "mobilenet_fp32.onnx"
INT8_ONNX = PROJECT / "models" / "int8_onnx" / "mobilenet_ort_int8.onnx"

print(FP32_ONNX)

Python: 3.13.15
Torch: 2.11.0+cu128
ONNX: 1.23.1
ORT: 1.30.0
timm: 1.0.30
Transformers: 4.57.6
ONNX Script: 0.7.2
CUDA: True
['AzureExecutionProvider', 'CPUExecutionProvider']
/content/drive/MyDrive/PTQ_MobileNet/models/fp32_onnx/mobilenet_fp32.onnx


## Export PyTorch model to ONNX

In [ ]:
# Non-TensorRT pipeline work
model_name = "mobilenetv3_small_050.lamb_in1k"
model = timm.create_model(model_name, pretrained=True)
model.eval()

# Find expected model input
config = timm.data.resolve_model_data_config(model)
print(config)

# PyTorch image models expect (batch_size, channels, height, width)
dummy_input = torch.randn(1, 3, 224, 224, dtype=torch.float32)

# Export the model to ONNX
torch.onnx.export(
    model,
    (dummy_input,),
    str(FP32_ONNX),
    input_names=["image"],
    output_names=["logits"],
    opset_version=17,
    dynamo=True,
    verify=True,
)

assert FP32_ONNX.exists()
print(f"Saved successfully: {FP32_ONNX}")
print(f"Size: {FP32_ONNX.stat().st_size / 1e6:.2f} MB")

W0929 22:41:58.336000 683 torch/onnx/_internal/exporter/_compat.py:133] Setting ONNX exporter to use operator set version 18 because the requested opset_version 17 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


{'input_size': (3, 224, 224), 'interpolation': 'bicubic', 'mean': (0.485, 0.456, 0.406), 'std': (0.229, 0.224, 0.225), 'crop_pct': 0.875, 'crop_mode': 'center'}
[torch.onnx] Obtain model graph for `MobileNetV3([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `MobileNetV3([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


/usr/lib/python3.13/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/onnxscript/version_converter/__init__.py", line 137, in call
    converted_proto = _c_api_utils.call_onnx_api(
        func=_partial_convert_version, model=model
    )
  File "/usr/local/lib/python3.13/dist-packages/onnxscript/version_converter/_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
  File "/usr/local/lib/python3.13/dist-packages/onnxscript/version_converter/__init__.py", line 132, in _partial_convert_version
    return onnx.version_converter.convert_version(
           ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^
        proto, target_version=self.target_version
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/onnx/version

[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
[torch.onnx] Check the ONNX model...
[torch.onnx] Check the ONNX model... ✅
[torch.onnx] Execute the model with ONNX Runtime...
[torch.onnx] Execute the model with ONNX Runtime... ✅
[torch.onnx] Verify output accuracy...
[torch.onnx] Verify output accuracy... ✅
Saved successfully: /content/drive/MyDrive/PTQ_MobileNet/models/fp32_onnx/mobilenet_fp32.onnx
Size: 0.28 MB


In [ ]:
# Static Quantization: both weights and activation layers are pre-computed before quantization
# Use cases: best for image / vision models
# Pros: faster inference, higher throughput
# Cons: more setup, require calibration dataset since activation layers
# How: pass calibrated inputs to observe activation layers. Using activation layer, find X_min and X_max.
# From this calculate 2 values: Scale and Zero-Point
# These values are directly passed to calculate the Quantized Value during the quantization process without calculating dyamically


'\n Static Quantization: both weights and activation layers are pre-computed before quantization\n Use cases: best for image / vision models\n Pros: faster inference, higher throughput\n Cons: more setup, require calibration dataset since activation layers\n How: pass calibrated inputs to observe activation layers. Using activation layer, find X_min and X_max.\n From this calculate 2 values: Scale and Zero-Point\n These values are directly passed to calculate the Quantized Value during the quantization process without calculating dyamically\n'

## Static Quantization pipeline for Non-TensorRT model


In [ ]:
# Static quantization pipeline
import onnxruntime
from onnxruntime.quantization import quantize_static, QuantType, QuantFormat, CalibrationDataReader, CalibrationMethod
from pathlib import Path
import numpy as np
import timm
from datasets import load_dataset

PROJECT = Path("/content/drive/MyDrive/PTQ_MobileNet")
FP32_ONNX = PROJECT / "models/fp32_onnx/mobilenet_fp32.onnx"
INT8_ONNX = PROJECT / "models/int8_onnx/mobilenet_int8.onnx.onnx"
INT8_ONNX.parent.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "mobilenetv3_small_050.lamb_in1k"
model = timm.create_model(MODEL_NAME, pretrained=True)
model.eval()

# Load beans calibration data: 100 training images
# Note these images are not used to train the MobileNet model, but they are are shown
# to the FP32 ONNX model so the quantizer can learn the activation layer pattern of the model
calib_ds = load_dataset("AI-Lab-Makerere/beans", split="train[:100]")

# Create MobileNet preprocessing pipeline
model = timm.create_model(MODEL_NAME, pretrained=True).eval() # .eval() is used for inference and to turn off dropout mechanism
data_cfg = timm.data.resolve_model_data_config(model) # read model's expected input settings: image size, crop settings, normalization, input channels, etc.
# build the preprocessing function which unpacks the dictionary key and value (e.g. batch_size=32)
# and transforms to the expected model input
transform = timm.data.create_transform(**data_cfg, is_training=False)

# Inpecting ONNX input
fp32_session = onnxruntime.InferenceSession(
    str(FP32_ONNX),
    providers=["CPUExecutionProvider"] # tells ONNX to use its CPU
)

input_name = fp32_session.get_inputs()[0].name
print("ONNX input:", input_name)
print("ONNX shape:", fp32_session.get_inputs()[0].shape)
print("ONNX type:", fp32_session.get_inputs()[0].type)

class MobileNetCalibrationReader(CalibrationDataReader):
    """Class that feeds calibration image to the quantizer.

      Args:
          CalibrationDataReader: the template of the data reader.

      Returns:
          None.
      """
    def __init__(self, dataset, input_name, transform):
        self.dataset = dataset  # Store 100 beans inside a reader
        self.input_name = input_name # ONNX input name
        self.transform = transform # Image scaling function
        self.index = 0

    def get_next(self):
        if self.index >= len(self.dataset): # No more data to read from
            return None

        # Ensure image has 3 RGB channels
        image = self.dataset[self.index]["image"].convert("RGB") # {"image": <PIL image object>, "labels": 1}

        # Add a dimension of size 1 before all dimensions
        # Previous image: (3,224,244)
        # New image: (1,3,224,224)
        tensor = self.transform(image).unsqueeze(0)

        self.index += 1

        return {self.input_name : tensor.numpy().astype(np.float32)}

# ONNX Runtime's static PTQ
# FP32 on calibration inputs, find activation layer's ranges, and calculate quantization parameters
quantize_static(
    model_input=str(FP32_ONNX),
    model_output=str(INT8_ONNX),
    calibration_data_reader=MobileNetCalibrationReader(
        calib_ds,
        input_name,
        transform
    ),
    # Quantize and DeQuantize
    # INT8
    # DeQuantizeLinear
    # FP32
    quant_format=QuantFormat.QDQ,
    weight_type=QuantType.QUInt8, # Quantized weights (usually -128 to 127)
    activation_type=QuantType.QUInt8, # Quantized activation value (0 to 255)
    calibrate_method=CalibrationMethod.MinMax, # Find the X_min and X_max to quantize the model
)

print(f"Saved INT8 Model:  {INT8_ONNX}")

ONNX input: image
ONNX shape: [1, 3, 224, 224]
ONNX type: tensor(float)


Saved INT8 Model:  /content/drive/MyDrive/PTQ_MobileNet/models/int8_onnx/mobilenet_int8.onnx.onnx


# TensorRT pipeline


In [ ]:
"""
TensorRT pipeline:
  - FP32 HF Model
  - Export to ONNX
  - TensorRT calibration + INT8 engine build
  - Benchmark / results
"""

'\nTensorRT pipeline:\n  - FP32 HF Model\n  - Export to ONNX\n  - TensorRT calibration + INT8 engine build\n  - Benchmark / results\n'

In [ ]:
from pathlib import Path
import numpy as np
import timm
from datasets import load_dataset

# Same as non-TensorRT for setup
PROJECT = Path("/content/drive/MyDrive/PTQ_MobileNet")
CALIB_NPY = PROJECT / "calibration/beans_calib_100.npy"
CALIB_NPY.parent.mkdir(parents=True, exist_ok=True)

model = timm.create_model(
    "mobilenetv3_small_050.lamb_in1k",
    pretrained=True,
).eval()

cfg = timm.data.resolve_model_data_config(model) # Determine the input size of the config
transform = timm.data.create_transform(**cfg, is_training=False)  # Transform function to scale the inputs

beans = load_dataset("AI-Lab-Makerere/beans", split="train[:100]")

calib = np.stack([
    transform(row["image"].convert("RGB")).numpy().astype(np.float32)
    for row in beans
]) # Stack each row of the beans dataset row-wise

np.save(CALIB_NPY, calib) # Save calibration dataset to CALIB_NPY path

print("Calibration shape:", calib.shape)
print("Saved:", CALIB_NPY)

Calibration shape: (100, 3, 224, 224)
Saved: /content/drive/MyDrive/PTQ_MobileNet/calibration/beans_calib_100.npy


## Fake ModelOpt INT8 QDQ




In [ ]:
# =============================================================================
# Model OPT essentially creates a runtime graph of the model
# How: It uses a calibration dataset to find the min, max, and scaling factors
# and stores them inside Q/DQ nodes. It doesn't turn an FP32 model into an INT8
# model.
# =============================================================================


%pip install -q nvidia-modelopt[onnx]

from pathlib import Path
from modelopt.onnx.quantization import quantize

FP32_ONNX = PROJECT / "models/fp32_onnx/mobilenet_fp32.onnx"
QDQ_ONNX = PROJECT / "models/modelopt_qdq/mobilenet_int8_qdq.onnx"
QDQ_ONNX.parent.mkdir(parents=True, exist_ok=True)

"""
  This step below is like the calibrated reader. It doesn't actually quantize, and
  only computes scaling factor, boundaries, and simulation.
"""
quantize(
    onnx_path=str(FP32_ONNX), # FP32 exported model
    quantize_mode="int8", # INT8 output model
    calibration_data=str(CALIB_NPY),  # Calibration dataset
    output_path=str(QDQ_ONNX),
)
print("Saved QDQ ONNX:", QDQ_ONNX)

### TensorRT Engine


In [ ]:
# =============================================================================
# Build a TensorRT INT8 engine from a Q/DQ ONNX model. Takes the graph and
# turn into a INT8 executable model
#
# Input:  mobilenet_int8_qdq.onnx  (ONNX with QuantizeLinear/DequantizeLinear
#         nodes already inserted, e.g. from post-training quantization or QAT)
# Output: mobilenet_int8.engine    (serialized TensorRT engine, ready to load
#         with trt.Runtime.deserialize_cuda_engine or trtexec --loadEngine)
# How: Fold weight Q into constants, Pattern matches Q -> DQ and fuses into an
# INT8 convolution, and Fuse ReLU layer
# The network is built in "strongly typed" mode: TensorRT takes each tensor's
# precision directly from the Q/DQ nodes in the ONNX graph instead of choosing
# precisions itself. That means no INT8/FP16 builder flags and no calibrator.
# Assumes PROJECT and QDQ_ONNX are defined in an earlier cell.
# =============================================================================

import tensorrt as trt
from pathlib import Path

# Where the finished engine will be written, and make sure the folder exists
ENGINE_PATH = PROJECT / "models/trt_engine/mobilenet_int8.engine"
ENGINE_PATH.parent.mkdir(parents=True, exist_ok=True)

# Logger that TensorRT uses to report messages; WARNING hides routine info logs.
# Switch to trt.Logger.VERBOSE to see which layers run in INT8.
logger = trt.Logger(trt.Logger.WARNING)

# The builder is the top-level object that creates networks and compiles engines
builder = trt.Builder(logger)

# Network creation flags are a bitmask. STRONGLY_TYPED makes TensorRT respect
# the precisions defined by the Q/DQ nodes rather than inferring its own.
flags = 1 << int(trt.NetworkDefinitionCreationFlag.STRONGLY_TYPED)

# Empty network definition that the ONNX parser will fill in
network = builder.create_network(flags)

# Parser that translates ONNX operators into TensorRT layers
parser = trt.OnnxParser(network, logger)

# Parse the Q/DQ ONNX file into the network; on failure, print each parser
# error and stop, since building a half-parsed network makes no sense
if not parser.parse_from_file(str(QDQ_ONNX)):
    for i in range(parser.num_errors):
        print(parser.get_error(i))
    raise RuntimeError("ONNX parse failed")

# Builder config holds the build settings (memory limits, flags, profiles)
config = builder.create_builder_config()

# Cap the scratch memory TensorRT may use while tuning kernels: 1 << 30 = 1 GB.
# This limits build-time memory, not the size of the final engine.
config.set_memory_pool_limit(trt.MemoryPoolType.WORKSPACE, 1 << 30)

# Deliberately NOT setting BuilderFlag.INT8 or FP16: in strongly typed mode
# the Q/DQ nodes already determine the precision of every tensor.

# Compile the network into an optimized, serialized engine (kernel selection,
# layer fusion, etc.). Returns None if the build fails.
serialized = builder.build_serialized_network(network, config)
if serialized is None:
    raise RuntimeError("TensorRT engine build failed")

# Write the serialized engine bytes to disk so it can be loaded later without
# rebuilding. Note that engines are tied to the GPU and TensorRT version used
# to build them.
ENGINE_PATH.write_bytes(bytes(serialized))
print("Saved engine:", ENGINE_PATH)

ModuleNotFoundError: No module named 'tensorrt'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
